# Molar Class II segmentation workflow

Source-only notebook for preprocessing, alignment, upscaling, and occlusal,
proximal, and cusp segmentation. Supply images and model checkpoints locally;
no datasets, trained weights, training logs, or stored outputs are included.

Run **Environment and paths**, then the desired prediction cell. Preparation
and training are separate workflows. Prediction preserves each input filename
stem and writes binary masks under `notebook_predictions/`; new checkpoints
go to `notebook_training/`. See [README.md](README.md) for setup and limitations.

## 1. Environment and paths

In [ ]:
import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications.resnet50 import preprocess_input

### Reproducibility and Image Input dimensions


In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
# Image input dimensions
input_height, input_width = 256, 256
input_channels = 3

In [ ]:
# Start Jupyter from this checkout, or set DENTAL_PROJECT_ROOT explicitly.
# PROJECT_ROOT contains the externally supplied image/training directories.
PROJECT_ROOT = Path(os.environ.get('DENTAL_PROJECT_ROOT', Path.cwd())).expanduser().resolve()
MODEL_DIR = Path(os.environ.get('DENTAL_MODEL_DIR', PROJECT_ROOT / 'models')).expanduser().resolve()
PREDICTION_OUTPUT_ROOT = PROJECT_ROOT / 'notebook_predictions'
TRAINING_OUTPUT_DIR = PROJECT_ROOT / 'notebook_training'
TARGET_SIZE = (input_width, input_height)
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Input project directory does not exist: {PROJECT_ROOT}')
os.chdir(PROJECT_ROOT)

## 2. Data preparation, alignment and upscaling

The original cropping, EDSR upscaling, image/mask alignment and validation-data
preparation code is retained. Set each section's input/output directories for
the required dataset before running it. Folder capitalization must match the
filesystem. EDSR requires `opencv-contrib-python` and the external `EDSR_x4.pb`;
its path is specified in each upscaling cell. The preparation sections use
their original output directories. Training and prediction below can use
already prepared data without rerunning this section.

### Processing student sample


In [ ]:
import os
import cv2
import numpy as np

def zoom_crop_tooth(img, target_size=(256, 256), pad=10, thresh_val=10):
    """
    Given a BGR image of a tooth on a black background, 
    find the tooth contour, crop to its bounding box ± padding, 
    and resize to target_size.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, thresh_val, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError("No contours found – check threshold or image.")
    tooth_cnt = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(tooth_cnt)
    x1 = max(x - pad, 0)
    y1 = max(y - pad, 0)
    x2 = min(x + w + pad, img.shape[1])
    y2 = min(y + h + pad, img.shape[0])
    crop = img[y1:y2, x1:x2]
    return cv2.resize(crop, target_size, interpolation=cv2.INTER_AREA)

def process_folder(input_folder, output_folder, target_size=(256,256)):
    os.makedirs(output_folder, exist_ok=True)
    for fname in os.listdir(input_folder):
        if not fname.lower().endswith(('.png','.jpg','.jpeg')):
            continue
        src_path = os.path.join(input_folder, fname)
        dst_path = os.path.join(output_folder, fname)
        img = cv2.imread(src_path)
        try:
            zoomed = zoom_crop_tooth(img, target_size=target_size)
            cv2.imwrite(dst_path, zoomed)
            print(f"[OK] {fname} → {output_folder}/{fname}")
        except Exception as e:
            print(f"[SKIP] {fname}: {e}")


if __name__ == "__main__":
    # --- User parameters ---
    input_folder  = "student_work"   # ← put your folder of full-tooth images here
    output_folder = "samples_sr"     # ← results will go here
    # ------------------------
    process_folder(input_folder, output_folder)


### Upscale Student Sample


In [ ]:
import os
import cv2
# upscaling
# ─── 1) SETUP ───────────────────────────────────────────────────────────────────
MODEL_PATH    = "../EDSR_x4.pb"
MODEL_NAME    = "edsr"    # lower-case
UPSCALE       = 4         # must be 4 for EDSR_x4

INPUT_FOLDER  = "samples_sr"
OUTPUT_FOLDER = "samples_stu"
FINAL_SIZE    = (256, 256)  # (height, width)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ─── 2) LOAD THE SR MODEL ───────────────────────────────────────────────────────
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(MODEL_PATH)
sr.setModel(MODEL_NAME, UPSCALE)

# ─── 3) PROCESS IMAGES ─────────────────────────────────────────────────────────
for fn in os.listdir(INPUT_FOLDER):
    if not fn.lower().endswith((".jpg", ".png")):
        continue

    img = cv2.imread(os.path.join(INPUT_FOLDER, fn))
    if img is None:
        print(f"Could not read {fn}, skipping.")
        continue

    # a) Upsample by the network
    up = sr.upsample(img)   # now (h0*UPSCALE, w0*UPSCALE)

    # b) Resize to the fixed FINAL_SIZE
    highres = cv2.resize(
        up,
        (FINAL_SIZE[1], FINAL_SIZE[0]),  # (width, height)
        interpolation=cv2.INTER_CUBIC
    )

    # c) Confirm dimensions
    h1, w1 = highres.shape[:2]
    if (h1, w1) != FINAL_SIZE:
        raise ValueError(f"Unexpected output size for {fn}: got {h1}×{w1}, expected {FINAL_SIZE[0]}×{FINAL_SIZE[1]}")

    # d) Save
    out_path = os.path.join(OUTPUT_FOLDER, fn)
    cv2.imwrite(out_path, highres)
    print(f"Saved SR {fn} → {OUTPUT_FOLDER} ({h1}×{w1})")


### Upscale Training Images: occlusal


In [ ]:
import os
import cv2
# upscaling
# ─── 1) SETUP ───────────────────────────────────────────────────────────────────
MODEL_PATH    = "../EDSR_x4.pb"
MODEL_NAME    = "edsr"    # lower-case
UPSCALE       = 4         # must be 4 for EDSR_x4


INPUT_FOLDER  = 'O_M_image'
OUTPUT_FOLDER = 'O_M_image_folder_up'
FINAL_SIZE    = (256, 256)  # (height, width)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ─── 2) LOAD THE SR MODEL ───────────────────────────────────────────────────────
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(MODEL_PATH)
sr.setModel(MODEL_NAME, UPSCALE)

# ─── 3) PROCESS IMAGES ─────────────────────────────────────────────────────────
for fn in os.listdir(INPUT_FOLDER):
    if not fn.lower().endswith((".jpg", ".png")):
        continue

    img = cv2.imread(os.path.join(INPUT_FOLDER, fn))
    if img is None:
        print(f"Could not read {fn}, skipping.")
        continue

    # a) Upsample by the network
    up = sr.upsample(img)   # now (h0*UPSCALE, w0*UPSCALE)

    # b) Resize to the fixed FINAL_SIZE
    highres = cv2.resize(
        up,
        (FINAL_SIZE[1], FINAL_SIZE[0]),  # (width, height)
        interpolation=cv2.INTER_CUBIC
    )

    # c) Confirm dimensions
    h1, w1 = highres.shape[:2]
    if (h1, w1) != FINAL_SIZE:
        raise ValueError(f"Unexpected output size for {fn}: got {h1}×{w1}, expected {FINAL_SIZE[0]}×{FINAL_SIZE[1]}")

    # d) Save
    out_path = os.path.join(OUTPUT_FOLDER, fn)
    cv2.imwrite(out_path, highres)
    print(f"Saved SR {fn} → {OUTPUT_FOLDER} ({h1}×{w1})")


### Upscale Training Images :Proximal


In [ ]:
import os
import cv2
# upscaling
# ─── 1) SETUP ───────────────────────────────────────────────────────────────────
MODEL_PATH    = "../EDSR_x4.pb"
MODEL_NAME    = "edsr"    # lower-case
UPSCALE       = 4         # must be 4 for EDSR_x4


INPUT_FOLDER  = 'P_M_image'
OUTPUT_FOLDER = 'P_M_image_folder_up'
FINAL_SIZE    = (256, 256)  # (height, width)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ─── 2) LOAD THE SR MODEL ───────────────────────────────────────────────────────
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(MODEL_PATH)
sr.setModel(MODEL_NAME, UPSCALE)

# ─── 3) PROCESS IMAGES ─────────────────────────────────────────────────────────
for fn in os.listdir(INPUT_FOLDER):
    if not fn.lower().endswith((".jpg", ".png")):
        continue

    img = cv2.imread(os.path.join(INPUT_FOLDER, fn))
    if img is None:
        print(f"Could not read {fn}, skipping.")
        continue

    # a) Upsample by the network
    up = sr.upsample(img)   # now (h0*UPSCALE, w0*UPSCALE)

    # b) Resize to the fixed FINAL_SIZE
    highres = cv2.resize(
        up,
        (FINAL_SIZE[1], FINAL_SIZE[0]),  # (width, height)
        interpolation=cv2.INTER_CUBIC
    )

    # c) Confirm dimensions
    h1, w1 = highres.shape[:2]
    if (h1, w1) != FINAL_SIZE:
        raise ValueError(f"Unexpected output size for {fn}: got {h1}×{w1}, expected {FINAL_SIZE[0]}×{FINAL_SIZE[1]}")

    # d) Save
    out_path = os.path.join(OUTPUT_FOLDER, fn)
    cv2.imwrite(out_path, highres)
    print(f"Saved SR {fn} → {OUTPUT_FOLDER} ({h1}×{w1})")


### Upscale Training Cavity Masks: Occlusal


In [ ]:
import os
import cv2
# upscaling
# ─── 1) SETUP ───────────────────────────────────────────────────────────────────
MODEL_PATH    = "../EDSR_x4.pb"
MODEL_NAME    = "edsr"    # lower-case
UPSCALE       = 4         # must be 4 for EDSR_x4


INPUT_FOLDER  = 'O_M_mask'
OUTPUT_FOLDER = 'O_M_mask_folder_up'
FINAL_SIZE    = (256, 256)  # (height, width)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ─── 2) LOAD THE SR MODEL ───────────────────────────────────────────────────────
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(MODEL_PATH)
sr.setModel(MODEL_NAME, UPSCALE)

# ─── 3) PROCESS IMAGES ─────────────────────────────────────────────────────────
for fn in os.listdir(INPUT_FOLDER):
    if not fn.lower().endswith((".jpg", ".png")):
        continue

    img = cv2.imread(os.path.join(INPUT_FOLDER, fn))
    if img is None:
        print(f"Could not read {fn}, skipping.")
        continue

    # a) Upsample by the network
    up = sr.upsample(img)   # now (h0*UPSCALE, w0*UPSCALE)

    # b) Resize to the fixed FINAL_SIZE
    highres = cv2.resize(
        up,
        (FINAL_SIZE[1], FINAL_SIZE[0]),  # (width, height)
        interpolation=cv2.INTER_CUBIC
    )

    # c) Confirm dimensions
    h1, w1 = highres.shape[:2]
    if (h1, w1) != FINAL_SIZE:
        raise ValueError(f"Unexpected output size for {fn}: got {h1}×{w1}, expected {FINAL_SIZE[0]}×{FINAL_SIZE[1]}")

    # d) Save
    out_path = os.path.join(OUTPUT_FOLDER, fn)
    cv2.imwrite(out_path, highres)
    print(f"Saved SR {fn} → {OUTPUT_FOLDER} ({h1}×{w1})")


### Upscale Training Cavity Masks: Proximal


In [ ]:
import os
import cv2
# upscaling
# ─── 1) SETUP ───────────────────────────────────────────────────────────────────
MODEL_PATH    = "../EDSR_x4.pb"
MODEL_NAME    = "edsr"    # lower-case
UPSCALE       = 4         # must be 4 for EDSR_x4


INPUT_FOLDER  = 'P_M_mask'
OUTPUT_FOLDER = 'P_M_mask_folder_up'
FINAL_SIZE    = (256, 256)  # (height, width)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ─── 2) LOAD THE SR MODEL ───────────────────────────────────────────────────────
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(MODEL_PATH)
sr.setModel(MODEL_NAME, UPSCALE)

# ─── 3) PROCESS IMAGES ─────────────────────────────────────────────────────────
for fn in os.listdir(INPUT_FOLDER):
    if not fn.lower().endswith((".jpg", ".png")):
        continue

    img = cv2.imread(os.path.join(INPUT_FOLDER, fn))
    if img is None:
        print(f"Could not read {fn}, skipping.")
        continue

    # a) Upsample by the network
    up = sr.upsample(img)   # now (h0*UPSCALE, w0*UPSCALE)

    # b) Resize to the fixed FINAL_SIZE
    highres = cv2.resize(
        up,
        (FINAL_SIZE[1], FINAL_SIZE[0]),  # (width, height)
        interpolation=cv2.INTER_CUBIC
    )

    # c) Confirm dimensions
    h1, w1 = highres.shape[:2]
    if (h1, w1) != FINAL_SIZE:
        raise ValueError(f"Unexpected output size for {fn}: got {h1}×{w1}, expected {FINAL_SIZE[0]}×{FINAL_SIZE[1]}")

    # d) Save
    out_path = os.path.join(OUTPUT_FOLDER, fn)
    cv2.imwrite(out_path, highres)
    print(f"Saved SR {fn} → {OUTPUT_FOLDER} ({h1}×{w1})")


### Upscale Training Cusp Masks


In [ ]:
import os
import cv2
# upscaling
# ─── 1) SETUP ───────────────────────────────────────────────────────────────────
MODEL_PATH    = "../EDSR_x4.pb"
MODEL_NAME    = "edsr"    # lower-case
UPSCALE       = 4         # must be 4 for EDSR_x4


INPUT_FOLDER  = 'M_CU_mask'
OUTPUT_FOLDER = 'M_CU_mask_folder_up'
FINAL_SIZE    = (256, 256)  # (height, width)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ─── 2) LOAD THE SR MODEL ───────────────────────────────────────────────────────
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(MODEL_PATH)
sr.setModel(MODEL_NAME, UPSCALE)

# ─── 3) PROCESS IMAGES ─────────────────────────────────────────────────────────
for fn in os.listdir(INPUT_FOLDER):
    if not fn.lower().endswith((".jpg", ".png")):
        continue

    img = cv2.imread(os.path.join(INPUT_FOLDER, fn))
    if img is None:
        print(f"Could not read {fn}, skipping.")
        continue

    # a) Upsample by the network
    up = sr.upsample(img)   # now (h0*UPSCALE, w0*UPSCALE)

    # b) Resize to the fixed FINAL_SIZE
    highres = cv2.resize(
        up,
        (FINAL_SIZE[1], FINAL_SIZE[0]),  # (width, height)
        interpolation=cv2.INTER_CUBIC
    )

    # c) Confirm dimensions
    h1, w1 = highres.shape[:2]
    if (h1, w1) != FINAL_SIZE:
        raise ValueError(f"Unexpected output size for {fn}: got {h1}×{w1}, expected {FINAL_SIZE[0]}×{FINAL_SIZE[1]}")

    # d) Save
    out_path = os.path.join(OUTPUT_FOLDER, fn)
    cv2.imwrite(out_path, highres)
    print(f"Saved SR {fn} → {OUTPUT_FOLDER} ({h1}×{w1})")

### Align training images, cavity masks and cusp masks

In [ ]:
# --- Preprocessing Reference Images, Masks and cusp mask for Vertical Alignment (Ellipse Fitting) ---
import os
import cv2
import numpy as np

# Directories
#ref_img_folder    = 'P_M_image_folder_up'
#ref_mask_folder   = 'P_M_mask_folder_up'
#ref_cusp_mask_folder ='M_CU_mask_folder_up'
#output_img_folder = 'prep_P_M_image_folder_up'
#output_mask_folder= 'prep_P_M_mask_folder_up'
#output_cusp_mask_folder='prep_P_M_cusp_mask_folder_up'

# Directories
#ref_img_folder    = 'P_M_image_folder_up_val'
#ref_mask_folder   = 'P_M_mask_folder_up_val'
#ref_cusp_mask_folder ='M_CU_mask_folder_up_val'
#output_img_folder = 'prep_P_M_image_folder_up_val'
#output_mask_folder= 'prep_P_M_mask_folder_up_val'
#output_cusp_mask_folder='prep_M_CU_mask_folder_up_val'


# Directories
ref_img_folder    = 'O_M_image_folder_up'
ref_mask_folder   = 'O_M_mask_folder_up'
ref_cusp_mask_folder ='M_CU_mask_folder_up'
output_img_folder = 'prep_O_M_image_folder_up'
output_mask_folder= 'prep_O_M_mask_folder_up'
output_cusp_mask_folder='prep_O_M_cusp_mask_folder_up'

# Directories
#ref_img_folder    = 'O_M_image_folder_up_val'
#ref_mask_folder   = 'O_M_mask_folder_up_val'
#ref_cusp_mask_folder ='M_CU_mask_folder_up_val'
#output_img_folder = 'prep_O_M_image_folder_up_val'
#output_mask_folder= 'prep_O_M_mask_folder_up_val'
#output_cusp_mask_folder='prep_O_M_CU_mask_folder_up_val'


# Uniform output size
TARGET_SIZE = (256, 256)

# Ensure output directories exist
os.makedirs(output_img_folder, exist_ok=True)
os.makedirs(output_mask_folder, exist_ok=True)
os.makedirs(output_cusp_mask_folder, exist_ok=True)


def compute_alignment_angle(mask_bin):
    """
    Use PCA on the cavity mask to find its principal axis and
    compute the rotation angle to align that axis vertically.
    """
    # Extract foreground coordinates
    ys, xs = np.nonzero(mask_bin)
    if len(xs) < 10:
        return 0.0
    # Center data
    x_mean, y_mean = xs.mean(), ys.mean()
    coords = np.vstack([xs - x_mean, ys - y_mean])  # shape (2, N)
    # Covariance and eigen decomposition
    cov = np.cov(coords)
    eig_vals, eig_vecs = np.linalg.eigh(cov)
    # Principal eigenvector
    principal = eig_vecs[:, np.argmax(eig_vals)]  # [vx, vy]
    vx, vy = principal[0], principal[1]
    # Angle between vector and vertical axis: phi = arctan2(vx, vy)
    angle_rad = np.arctan2(vx, vy)
    # Rotation needed to bring principal axis to vertical: -phi
    rot_deg = -np.degrees(angle_rad)
    return rot_deg


def preprocess_references():
    """
    Resize, align and save reference images and masks so that the cavity
    major axis is vertical.
    """
    for fname in sorted(os.listdir(ref_img_folder)):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        base, ext = os.path.splitext(fname)
        img_path  = os.path.join(ref_img_folder, fname)
        mask_path = os.path.join(ref_mask_folder, base + '.png')
        cusp_mask_path = os.path.join(ref_cusp_mask_folder, base + '.png')
        
        if not os.path.exists(mask_path):
            print(f"Warning: mask for {fname} not found, skipping.")
            continue

        # Load and resize
        img = cv2.resize(cv2.imread(img_path), TARGET_SIZE, interpolation=cv2.INTER_CUBIC)
        img_cusp=cv2.resize(cv2.imread(cusp_mask_path), TARGET_SIZE, interpolation=cv2.INTER_CUBIC)
        mask_gray = cv2.resize(cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE), TARGET_SIZE,
                               interpolation=cv2.INTER_NEAREST)
        # Binary mask
        _, mask_bin = cv2.threshold(mask_gray, 0, 255, cv2.THRESH_BINARY)

        # Keep only largest connected component
        num_labels, labels = cv2.connectedComponents(mask_bin)
        if num_labels > 1:
            counts = np.bincount(labels.flatten())
            counts[0] = 0
            largest = counts.argmax()
            mask_cc = (labels == largest).astype(np.uint8) * 255
        else:
            mask_cc = mask_bin

        # Compute rotation angle
        angle = compute_alignment_angle(mask_cc)

        # Rotate image and mask around center
        M = cv2.getRotationMatrix2D((TARGET_SIZE[0]/2, TARGET_SIZE[1]/2), angle, 1.0)
        img_rot  = cv2.warpAffine(img,  M, TARGET_SIZE,
                                  flags=cv2.INTER_CUBIC,
                                  borderMode=cv2.BORDER_CONSTANT,
                                  borderValue=(0,0,0))
        img_cusp_rot=cv2.warpAffine(img_cusp,  M, TARGET_SIZE,
                                  flags=cv2.INTER_CUBIC,
                                  borderMode=cv2.BORDER_CONSTANT,
                                  borderValue=(0,0,0))
        mask_rot = cv2.warpAffine(mask_cc, M, TARGET_SIZE,
                                  flags=cv2.INTER_NEAREST,
                                  borderMode=cv2.BORDER_CONSTANT,
                                  borderValue=0)

        # Save aligned outputs
        cv2.imwrite(os.path.join(output_img_folder,  base + ext), img_rot)
        cv2.imwrite(os.path.join(output_cusp_mask_folder, base + '.png'), img_cusp_rot)
        cv2.imwrite(os.path.join(output_mask_folder, base + '.png'), mask_rot)

    print("Preprocessing images complete. Outputs in:")
    print(f"  Images: {output_img_folder}\n  Masks: {output_mask_folder}\n Cusp Masks:{output_cusp_mask_folder}")
    


if __name__ == '__main__':
    preprocess_references()
  


### Upscale Validation Images


In [ ]:
import os
import cv2
# upscaling
# ─── 1) SETUP ───────────────────────────────────────────────────────────────────
MODEL_PATH    = "EDSR_x4.pb"
MODEL_NAME    = "edsr"    # lower-case
UPSCALE       = 4         # must be 4 for EDSR_x4


INPUT_FOLDER  = 'val_images_folder'
OUTPUT_FOLDER = 'val_images_folder_up'
FINAL_SIZE    = (256, 256)  # (height, width)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ─── 2) LOAD THE SR MODEL ───────────────────────────────────────────────────────
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(MODEL_PATH)
sr.setModel(MODEL_NAME, UPSCALE)

# ─── 3) PROCESS IMAGES ─────────────────────────────────────────────────────────
for fn in os.listdir(INPUT_FOLDER):
    if not fn.lower().endswith((".jpg", ".png")):
        continue

    img = cv2.imread(os.path.join(INPUT_FOLDER, fn))
    if img is None:
        print(f"Could not read {fn}, skipping.")
        continue

    # a) Upsample by the network
    up = sr.upsample(img)   # now (h0*UPSCALE, w0*UPSCALE)

    # b) Resize to the fixed FINAL_SIZE
    highres = cv2.resize(
        up,
        (FINAL_SIZE[1], FINAL_SIZE[0]),  # (width, height)
        interpolation=cv2.INTER_CUBIC
    )

    # c) Confirm dimensions
    h1, w1 = highres.shape[:2]
    if (h1, w1) != FINAL_SIZE:
        raise ValueError(f"Unexpected output size for {fn}: got {h1}×{w1}, expected {FINAL_SIZE[0]}×{FINAL_SIZE[1]}")

    # d) Save
    out_path = os.path.join(OUTPUT_FOLDER, fn)
    cv2.imwrite(out_path, highres)
    print(f"Saved SR {fn} → {OUTPUT_FOLDER} ({h1}×{w1})")


### Upscale Validation Cavity masks


In [ ]:
import os
import cv2
# upscaling
# ─── 1) SETUP ───────────────────────────────────────────────────────────────────
MODEL_PATH    = "EDSR_x4.pb"
MODEL_NAME    = "edsr"    # lower-case
UPSCALE       = 4         # must be 4 for EDSR_x4


INPUT_FOLDER  = 'val_masks_folder'
OUTPUT_FOLDER = 'val_masks_folder_up'
FINAL_SIZE    = (256, 256)  # (height, width)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ─── 2) LOAD THE SR MODEL ───────────────────────────────────────────────────────
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(MODEL_PATH)
sr.setModel(MODEL_NAME, UPSCALE)

# ─── 3) PROCESS IMAGES ─────────────────────────────────────────────────────────
for fn in os.listdir(INPUT_FOLDER):
    if not fn.lower().endswith((".jpg", ".png")):
        continue

    img = cv2.imread(os.path.join(INPUT_FOLDER, fn))
    if img is None:
        print(f"Could not read {fn}, skipping.")
        continue

    # a) Upsample by the network
    up = sr.upsample(img)   # now (h0*UPSCALE, w0*UPSCALE)

    # b) Resize to the fixed FINAL_SIZE
    highres = cv2.resize(
        up,
        (FINAL_SIZE[1], FINAL_SIZE[0]),  # (width, height)
        interpolation=cv2.INTER_CUBIC
    )

    # c) Confirm dimensions
    h1, w1 = highres.shape[:2]
    if (h1, w1) != FINAL_SIZE:
        raise ValueError(f"Unexpected output size for {fn}: got {h1}×{w1}, expected {FINAL_SIZE[0]}×{FINAL_SIZE[1]}")

    # d) Save
    out_path = os.path.join(OUTPUT_FOLDER, fn)
    cv2.imwrite(out_path, highres)
    print(f"Saved SR {fn} → {OUTPUT_FOLDER} ({h1}×{w1})")


### Upscale Validation Cusp masks


In [ ]:
import os
import cv2
# upscaling
# ─── 1) SETUP ───────────────────────────────────────────────────────────────────
MODEL_PATH    = "EDSR_x4.pb"
MODEL_NAME    = "edsr"    # lower-case
UPSCALE       = 4         # must be 4 for EDSR_x4


INPUT_FOLDER  = 'val_cusp_molar_mask'
OUTPUT_FOLDER = 'val_cusp_molar_mask_up'
FINAL_SIZE    = (256, 256)  # (height, width)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ─── 2) LOAD THE SR MODEL ───────────────────────────────────────────────────────
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(MODEL_PATH)
sr.setModel(MODEL_NAME, UPSCALE)

# ─── 3) PROCESS IMAGES ─────────────────────────────────────────────────────────
for fn in os.listdir(INPUT_FOLDER):
    if not fn.lower().endswith((".jpg", ".png")):
        continue

    img = cv2.imread(os.path.join(INPUT_FOLDER, fn))
    if img is None:
        print(f"Could not read {fn}, skipping.")
        continue

    # a) Upsample by the network
    up = sr.upsample(img)   # now (h0*UPSCALE, w0*UPSCALE)

    # b) Resize to the fixed FINAL_SIZE
    highres = cv2.resize(
        up,
        (FINAL_SIZE[1], FINAL_SIZE[0]),  # (width, height)
        interpolation=cv2.INTER_CUBIC
    )

    # c) Confirm dimensions
    h1, w1 = highres.shape[:2]
    if (h1, w1) != FINAL_SIZE:
        raise ValueError(f"Unexpected output size for {fn}: got {h1}×{w1}, expected {FINAL_SIZE[0]}×{FINAL_SIZE[1]}")

    # d) Save
    out_path = os.path.join(OUTPUT_FOLDER, fn)
    cv2.imwrite(out_path, highres)
    print(f"Saved SR {fn} → {OUTPUT_FOLDER} ({h1}×{w1})")


### Processing Validation Images,  Cavity Masks and Cusp mask for Vertical Alignment


In [ ]:
# --- Preprocessing Validation Images,  Masks and cusp mask for Vertical Alignment (Ellipse Fitting) ---
import os
import cv2
import numpy as np

# Directories
ref_img_folder    = 'P_M_image_folder_up_val'
ref_mask_folder   = 'P_M_mask_folder_up_val'
ref_cusp_mask_folder ='M_CU_mask_folder_up_val'
output_img_folder = 'prep_P_M_image_folder_up_val'
output_mask_folder= 'prep_P_M_mask_folder_up_val'
output_cusp_mask_folder='prep_M_CU_mask_folder_up_val'

# Uniform output size
TARGET_SIZE = (256, 256)

# Ensure output directories exist
os.makedirs(output_img_folder, exist_ok=True)
os.makedirs(output_mask_folder, exist_ok=True)


def compute_alignment_angle(mask_bin):
    """
    Use PCA on the cavity mask to find its principal axis and
    compute the rotation angle to align that axis vertically.
    """
    # Extract foreground coordinates
    ys, xs = np.nonzero(mask_bin)
    if len(xs) < 10:
        return 0.0
    # Center data
    x_mean, y_mean = xs.mean(), ys.mean()
    coords = np.vstack([xs - x_mean, ys - y_mean])  # shape (2, N)
    # Covariance and eigen decomposition
    cov = np.cov(coords)
    eig_vals, eig_vecs = np.linalg.eigh(cov)
    # Principal eigenvector
    principal = eig_vecs[:, np.argmax(eig_vals)]  # [vx, vy]
    vx, vy = principal[0], principal[1]
    # Angle between vector and vertical axis: phi = arctan2(vx, vy)
    angle_rad = np.arctan2(vx, vy)
    # Rotation needed to bring principal axis to vertical: -phi
    rot_deg = -np.degrees(angle_rad)
    return rot_deg


def preprocess_references():
    """
    Resize, align and save reference images and masks so that the cavity
    major axis is vertical.
    """
    for fname in sorted(os.listdir(ref_img_folder)):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        base, ext = os.path.splitext(fname)
        img_path  = os.path.join(ref_img_folder, fname)
        mask_path = os.path.join(ref_mask_folder, base + '.png')
        val_cusp_mask_path = os.path.join(ref_cusp_mask_folder, base + '.png')
        if not os.path.exists(mask_path):
            print(f"Warning: mask for {fname} not found, skipping.")
            continue

        # Load and resize
        img = cv2.resize(cv2.imread(img_path), TARGET_SIZE, interpolation=cv2.INTER_CUBIC)
        val_img_cusp=cv2.resize(cv2.imread(val_cusp_mask_path), TARGET_SIZE, interpolation=cv2.INTER_CUBIC)
        mask_gray = cv2.resize(cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE), TARGET_SIZE,
                               interpolation=cv2.INTER_NEAREST)
        # Binary mask
        _, mask_bin = cv2.threshold(mask_gray, 0, 255, cv2.THRESH_BINARY)

        # Keep only largest connected component
        num_labels, labels = cv2.connectedComponents(mask_bin)
        if num_labels > 1:
            counts = np.bincount(labels.flatten())
            counts[0] = 0
            largest = counts.argmax()
            mask_cc = (labels == largest).astype(np.uint8) * 255
        else:
            mask_cc = mask_bin

        # Compute rotation angle
        angle = compute_alignment_angle(mask_cc)

        # Rotate image and mask around center
        M = cv2.getRotationMatrix2D((TARGET_SIZE[0]/2, TARGET_SIZE[1]/2), angle, 1.0)
        img_rot  = cv2.warpAffine(img,  M, TARGET_SIZE,
                                  flags=cv2.INTER_CUBIC,
                                  borderMode=cv2.BORDER_CONSTANT,
                                  borderValue=(0,0,0))
        val_img_cusp_rot=cv2.warpAffine(val_img_cusp,  M, TARGET_SIZE,
                                  flags=cv2.INTER_CUBIC,
                                  borderMode=cv2.BORDER_CONSTANT,
                                  borderValue=(0,0,0))
        mask_rot = cv2.warpAffine(mask_cc, M, TARGET_SIZE,
                                  flags=cv2.INTER_NEAREST,
                                  borderMode=cv2.BORDER_CONSTANT,
                                  borderValue=0)

        # Save aligned outputs
        cv2.imwrite(os.path.join(output_img_folder,  base + ext), img_rot)
        cv2.imwrite(os.path.join(output_cusp_mask_folder, base + '.png'), val_img_cusp_rot)
        cv2.imwrite(os.path.join(output_mask_folder, base + '.png'), mask_rot)

    print("Preprocessing complete. Outputs in:")
    print(f"  Images: {output_img_folder}\n  Masks: {output_mask_folder}\n val Cusp Masks:{output_cusp_mask_folder}")

if __name__ == '__main__':
    preprocess_references()


## 3. Predict masks with externally supplied checkpoints

Run the environment/path cells first, then any prediction cell independently.
Each cell loads its checkpoint from `MODEL_DIR` and predicts all supported
images in `samples_stu_O/` or `samples_stu_P/`. It does not require prior results.
Occlusal input uses OpenCV BGR followed by ResNet50 preprocessing. Proximal and
cusp input uses RGB divided by 255. Masks use a threshold of 0.5 at 256 × 256.

### Occlusal cavity

In [ ]:
model = load_model(str(MODEL_DIR / 'O_unet_refined.h5'), compile=False)
samples_folder = PROJECT_ROOT / 'samples_stu_O'
pred_folder = PREDICTION_OUTPUT_ROOT / 'pred_O_M_masks_folder'
pred_folder.mkdir(parents=True, exist_ok=True)
TARGET_SIZE = (256, 256)

for fname in sorted(os.listdir(samples_folder)):
    if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    base    = os.path.splitext(fname)[0]
    sample_id = base
    path    = os.path.join(samples_folder, fname)
    img_bgr = cv2.imread(path)                       # BGR (matches training)
    img_bgr = cv2.resize(img_bgr, TARGET_SIZE)

    # for display only — convert to RGB
    disp    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # for the model — use BGR + preprocess_input (matches training)
    inp     = preprocess_input(img_bgr.astype('float32'))

    prob = model.predict(inp[np.newaxis, ...])[0, ..., 0]
    mask = (prob > 0.5).astype(np.uint8) * 255

    cv2.imwrite(os.path.join(pred_folder, sample_id + '_mask.png'), mask)

    overlay = disp.copy()
    overlay[mask > 0] = (0, 255, 0)

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(disp);          axs[0].set_title('Original');       axs[0].axis('off')
    axs[1].imshow(mask, cmap='gray'); axs[1].set_title('Predicted Mask'); axs[1].axis('off')
    axs[2].imshow(overlay);       axs[2].set_title('Overlay');        axs[2].axis('off')
    fig.suptitle(sample_id)
    plt.show()

### Proximal cavity

In [ ]:
model = load_model(str(MODEL_DIR / 'P_unet_refined.h5'), compile=False)
samples_folder = PROJECT_ROOT / 'samples_stu_P'
pred_folder = PREDICTION_OUTPUT_ROOT / 'pred_P_M_masks_folder'
pred_folder.mkdir(parents=True, exist_ok=True)
TARGET_SIZE = (256, 256)

for fname in sorted(os.listdir(samples_folder)):
    if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    base    = os.path.splitext(fname)[0]
    sample_id = base
    path    = os.path.join(samples_folder, fname)
    img_bgr = cv2.imread(path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    disp    = cv2.resize(img_rgb, TARGET_SIZE)
    inp     = disp / 255.0

    # predict & threshold
    prob = model.predict(inp[np.newaxis,...])[0,...,0]
    mask = (prob > 0.5).astype(np.uint8) * 255

    # save binary mask
    cv2.imwrite(os.path.join(pred_folder, sample_id + '_mask.png'), mask)

    # display original + mask + overlay
    overlay = disp.copy()
    overlay[mask>0] = (0,255,0)  # green overlay

    fig, axs = plt.subplots(1,3, figsize=(12,4))
    axs[0].imshow(disp)
    axs[0].set_title('Original'); axs[0].axis('off')
    axs[1].imshow(mask, cmap='gray')
    axs[1].set_title('Predicted Mask'); axs[1].axis('off')
    axs[2].imshow(overlay)
    axs[2].set_title('Overlay'); axs[2].axis('off')
    fig.suptitle(sample_id)
    plt.show()

### Cusps

In [ ]:
model = load_model(str(MODEL_DIR / 'CU_unet_refined.h5'), compile=False)
samples_folder = PROJECT_ROOT / 'samples_stu_O'
pred_folder = PREDICTION_OUTPUT_ROOT / 'pred_O_M_cusp_molar_mask'
pred_folder.mkdir(parents=True, exist_ok=True)
TARGET_SIZE = (256, 256)

for fname in sorted(os.listdir(samples_folder)):
    if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    base    = os.path.splitext(fname)[0]
    sample_id = base
    path    = os.path.join(samples_folder, fname)
    img_bgr = cv2.imread(path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    disp    = cv2.resize(img_rgb, TARGET_SIZE)
    inp     = disp / 255.0

    # predict & threshold
    prob = model.predict(inp[np.newaxis,...])[0,...,0]
    mask = (prob > 0.5).astype(np.uint8) * 255

    # save binary mask
    cv2.imwrite(os.path.join(pred_folder, sample_id + '_mask.png'), mask)

    # display original + mask + overlay
    overlay = disp.copy()
    overlay[mask>0] = (0,255,0)  # green overlay

    fig, axs = plt.subplots(1,3, figsize=(12,4))
    axs[0].imshow(disp)
    axs[0].set_title('Original'); axs[0].axis('off')
    axs[1].imshow(mask, cmap='gray')
    axs[1].set_title('Predicted Mask'); axs[1].axis('off')
    axs[2].imshow(overlay)
    axs[2].set_title('Overlay'); axs[2].axis('off')
    fig.suptitle(sample_id)
    plt.show()

## 4. Model training

These sections retain the architectures, losses, augmentation settings, and
training schedules. Run the shared generators first, then the complete section
for the selected model. New training saves to `notebook_training/`; prediction
loads checkpoints from `MODEL_DIR`. Move or select trained weights explicitly.

The unit-scaled and ResNet generators have separate names. The original
unit-scaled generator's BGR training convention and the RGB inference convention
are preserved. Check this distinction before retraining or replacing weights.
Saved training histories have been cleared; a new run produces its own metrics.

### Shared image/mask resizing

In [ ]:
def resize_pair(img, mask, size=(input_width, input_height)):
    """Resize image and mask to given size."""
    img_resized  = cv2.resize(img, size, interpolation=cv2.INTER_LINEAR)
    mask_resized = cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST)
    return img_resized, mask_resized

### Unit-scaled generator for proximal and cusp training

In [ ]:
class UnitScaleSegmentationDataGenerator(Sequence):
    def __init__(self, imgs_dir, masks_dir, batch_size, image_datagen, mask_datagen):
        self.img_files = sorted([f for f in os.listdir(imgs_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.imgs_dir  = imgs_dir
        self.masks_dir = masks_dir
        self.batch_size= batch_size
        self.img_gen   = image_datagen
        self.mask_gen  = mask_datagen

    def __len__(self):
        return int(np.ceil(len(self.img_files) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_files = self.img_files[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_imgs, batch_masks = [], []

        for fname in batch_files:
            name, _ = os.path.splitext(fname)
            img_path  = os.path.join(self.imgs_dir, fname)
            mask_path = os.path.join(self.masks_dir, name + '.png')

            # Load
            img  = cv2.imread(img_path, cv2.IMREAD_COLOR)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            # Resize to fixed input size
            img, mask = resize_pair(img, mask)

            # Apply augmentation (identity for validation)
            transform = self.img_gen.get_random_transform(img.shape)
            img_aug   = self.img_gen.apply_transform(img, transform)
            mask_aug  = self.mask_gen.apply_transform(mask[...,None], transform)[...,0]

            # Final processing
            img_aug   = img_aug.astype('float32') / 255.0
            # if u r using ResNet50 then do the following
            #img_aug = preprocess_input(img_aug.astype('float32'))

            mask_aug  = (mask_aug > 127).astype(np.float32)[...,None]
            
            batch_imgs.append(img_aug)
            batch_masks.append(mask_aug)

        # Stack; all shapes now uniform
        X = np.stack(batch_imgs, axis=0)
        Y = np.stack(batch_masks, axis=0)
        return X, Y

    def on_epoch_end(self):
        np.random.shuffle(self.img_files)

### ResNet50 generator for occlusal training

In [ ]:
class ResNetSegmentationDataGenerator(Sequence):
    def __init__(self, imgs_dir, masks_dir, batch_size, image_datagen, mask_datagen):
        self.img_files = sorted([f for f in os.listdir(imgs_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.imgs_dir  = imgs_dir
        self.masks_dir = masks_dir
        self.batch_size= batch_size
        self.img_gen   = image_datagen
        self.mask_gen  = mask_datagen

    def __len__(self):
        return int(np.ceil(len(self.img_files) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_files = self.img_files[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_imgs, batch_masks = [], []

        for fname in batch_files:
            name, _ = os.path.splitext(fname)
            img_path  = os.path.join(self.imgs_dir, fname)
            mask_path = os.path.join(self.masks_dir, name + '.png')

            # Load
            img  = cv2.imread(img_path, cv2.IMREAD_COLOR)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            # Resize to fixed input size
            img, mask = resize_pair(img, mask)

            # Apply augmentation (identity for validation)
            transform = self.img_gen.get_random_transform(img.shape)
            img_aug   = self.img_gen.apply_transform(img, transform)
            mask_aug  = self.mask_gen.apply_transform(mask[...,None], transform)[...,0]

            # Final processing
            #img_aug   = img_aug.astype('float32') / 255.0
            # if u r using ResNet50 then do the following
            img_aug = preprocess_input(img_aug.astype('float32'))

            mask_aug  = (mask_aug > 127).astype(np.float32)[...,None]
            
            batch_imgs.append(img_aug)
            batch_masks.append(mask_aug)

        # Stack; all shapes now uniform
        X = np.stack(batch_imgs, axis=0)
        Y = np.stack(batch_masks, axis=0)
        return X, Y

    def on_epoch_end(self):
        np.random.shuffle(self.img_files)

### Proximal refined U-Net

### Directory Definitions: Proximal


In [ ]:
imgs_dir  = 'prep_P_M_image_folder_up'
masks_dir = 'prep_P_M_mask_folder_up'
cusp_masks_dir = 'prep_P_M_cusp_mask_folder_up'
val_imgs_dir  = 'prep_P_M_image_folder_up_val'
val_masks_dir = 'prep_P_M_mask_folder_up_val'
val_cusp_masks_dir = 'prep_M_CU_mask_folder_up_val'
samples_dir    = 'samples_stu_P'
# Output folder for predicted masks
pred_masks_dir = 'pred_P_M_masks_folder'
pred_cusp_mask_dir = 'pred_M_cusp_molar_mask'
# Create output dir if needed
os.makedirs(pred_masks_dir, exist_ok=True)
os.makedirs(pred_cusp_mask_dir, exist_ok=True)

### Augmentation parameters for proximal cavity


In [ ]:
data_gen_args = dict(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=5,
    brightness_range=(0.8, 1.2),
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

batch_size = 4
val_batch_size= 4


### Defining Keras ImageDataGenerators for proximal cavity


In [ ]:
# Training generators with augmentation
train_image_datagen = ImageDataGenerator(**data_gen_args)
train_mask_datagen  = ImageDataGenerator(**data_gen_args)
train_cusp_mask_datagen = ImageDataGenerator(**data_gen_args)

#train_image_datagen = ImageDataGenerator()
#train_mask_datagen  = ImageDataGenerator()
# Validation generators without augmentation
val_image_datagen   = ImageDataGenerator()
val_mask_datagen    = ImageDataGenerator()
val_cusp_mask_datagen = ImageDataGenerator()

### Instantiate Generators for proximal cavity


In [ ]:
# %%
train_gen = UnitScaleSegmentationDataGenerator(
    imgs_dir, masks_dir, batch_size,
    train_image_datagen, train_mask_datagen
)
val_gen   = UnitScaleSegmentationDataGenerator(
    val_imgs_dir, val_masks_dir, val_batch_size,
    val_image_datagen, val_mask_datagen
)

# Sanity checks
x_val, y_val = val_gen[0]
x_tr , y_tr  = train_gen[0]
print("Validation batch shapes:", x_val.shape, y_val.shape)
print("Training   batch shapes:", x_tr.shape , y_tr.shape)

### Define Refined U-Net segmentation Model for Proximal cavity prediction


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, MaxPooling2D,
    Dropout, concatenate, BatchNormalization, Activation
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import backend as K

# ─── CUSTOM LOSSES & METRICS ───────────────────────────────────────────────────
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.*intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

#def bce_dice_edge_loss(y_true, y_pred):
#    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
#    d   = dice_loss(y_true, y_pred)
#    # Sobel edge loss
#    gy_true, gx_true = tf.image.sobel_edges(y_true)[...,0], tf.image.sobel_edges(y_true)[...,1]
#    gy_pred, gx_pred = tf.image.sobel_edges(y_pred)[...,0], tf.image.sobel_edges(y_pred)[...,1]
#    edge_true = tf.sqrt(gx_true**2 + gy_true**2)
#    edge_pred = tf.sqrt(gx_pred**2 + gy_pred**2)
#    edge_loss = K.mean(K.abs(edge_true - edge_pred))
#    return bce + d + 0.5*edge_loss
def bce_dice_edge_loss(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    d   = dice_loss(y_true, y_pred)

    # sobel_edges expects 4D (B,H,W,C); returns (B,H,W,C,2) where last axis = [dy, dx]
    sob_true = tf.image.sobel_edges(y_true)
    sob_pred = tf.image.sobel_edges(y_pred)

    dy_true, dx_true = sob_true[..., 0], sob_true[..., 1]
    dy_pred, dx_pred = sob_pred[..., 0], sob_pred[..., 1]

    # +1e-6 inside sqrt keeps the gradient finite at zero-gradient regions
    edge_true = tf.sqrt(dx_true**2 + dy_true**2 + 1e-6)
    edge_pred = tf.sqrt(dx_pred**2 + dy_pred**2 + 1e-6)

    edge_loss = K.mean(K.abs(edge_true - edge_pred))
    return bce + d + 0.5 * edge_loss


def iou_metric(y_true, y_pred, smooth=1):
    intersection = K.sum(K.abs(y_true * y_pred))
    union = K.sum(y_true) + K.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

# ─── REFINED U-NET ARCHITECTURE ────────────────────────────────────────────────
def P_build_refined_unet(input_size=(256,256,3)):
    inputs = Input(input_size)
    def enc_block(x, filters):
        x = Conv2D(filters,3,padding="same")(x)
        x = BatchNormalization()(x); x = Activation("relu")(x)
        x = Conv2D(filters,3,padding="same")(x)
        x = BatchNormalization()(x); x = Activation("relu")(x)
        p = MaxPooling2D()(x)
        return x, p

    c1,p1 = enc_block(inputs,64)
    c2,p2 = enc_block(p1,128)
    c3,p3 = enc_block(p2,256)
    c4,p4 = enc_block(p3,512)

    b = Conv2D(1024,3,padding="same")(p4)
    b = BatchNormalization()(b); b = Activation("relu")(b)
    b = Dropout(0.5)(b)
    b = Conv2D(1024,3,padding="same")(b)
    b = BatchNormalization()(b); b = Activation("relu")(b)

    def dec_block(x, skip, filters):
        x = Conv2DTranspose(filters,2,strides=2,padding="same")(x)
        x = concatenate([x,skip])
        x = Conv2D(filters,3,padding="same")(x)
        x = BatchNormalization()(x); x = Activation("relu")(x)
        x = Conv2D(filters,3,padding="same")(x)
        x = BatchNormalization()(x); x = Activation("relu")(x)
        return x

    d1 = dec_block(b, c4, 512)
    d2 = dec_block(d1, c3, 256)
    d3 = dec_block(d2, c2, 128)
    d4 = dec_block(d3, c1,  64)

    outputs = Conv2D(1,1,activation="sigmoid")(d4)
    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(1e-4),
        loss=bce_dice_edge_loss,
        metrics=[dice_coef, iou_metric]
    )
    return model

### Fitting Refined U-Net segmentation Model for Proximal cavity prediction


In [ ]:
TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ─── TRAINING ──────────────────────────────────────────────────────────────────
model = P_build_refined_unet()
model.summary()


checkpoint = ModelCheckpoint(
    str(TRAINING_OUTPUT_DIR / "P_unet_refined.h5"), save_best_only=True,
    monitor="val_iou_metric", mode="max", verbose=1
)
earlystop = EarlyStopping(
    monitor="val_iou_metric", mode="max",
    patience=20, restore_best_weights=True, verbose=1
)

history = model.fit(
     train_gen,
     validation_data=val_gen,
     epochs=500,
     callbacks=[checkpoint, earlystop]
 )

### Occlusal U-Net with ResNet50 encoder

### Directory Definitions: Occlusal


In [ ]:

imgs_dir  = 'prep_O_M_image_folder_up'
masks_dir = 'prep_O_M_mask_folder_up'
cusp_masks_dir = 'prep_O_M_cusp_mask_folder_up'
val_imgs_dir  = 'prep_O_M_image_folder_up_val'
val_masks_dir = 'prep_O_M_mask_folder_up_val'
val_cusp_masks_dir = 'prep_O_M_CU_mask_folder_up_val'
samples_dir    = 'samples_stu_O'
# Output folder for predicted masks
pred_masks_dir = 'pred_O_M_masks_folder'
pred_cusp_mask_dir = 'pred_O_M_cusp_molar_mask'
# Create output dir if needed
os.makedirs(pred_masks_dir, exist_ok=True)
os.makedirs(pred_cusp_mask_dir, exist_ok=True)

### Augmentation parameters for Occlusal cavity


In [ ]:
image_datagen = dict(
    rotation_range=15,           # was 45, too much for teeth
    width_shift_range=0.1,       # was 0.2
    height_shift_range=0.1,      # was 0.2
    shear_range=0.05,            # was 0.15
    zoom_range=0.1,              # was 0.2
    horizontal_flip=True,
    vertical_flip=False,         # teeth have a clear up/down
    fill_mode='reflect',
    brightness_range=[0.9, 1.1], # was [0.8, 1.2]
)

mask_datagen = dict(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    vertical_flip=False,
    fill_mode='reflect',
)

batch_size = 4
val_batch_size= 4

### Define the final occlusal architecture

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K

def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.*intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_dice_edge_loss(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    d   = dice_loss(y_true, y_pred)

    sob_true = tf.image.sobel_edges(y_true)
    sob_pred = tf.image.sobel_edges(y_pred)

    dy_true, dx_true = sob_true[..., 0], sob_true[..., 1]
    dy_pred, dx_pred = sob_pred[..., 0], sob_pred[..., 1]

    edge_true = tf.sqrt(dx_true**2 + dy_true**2 + 1e-7)   # <-- add epsilon
    edge_pred = tf.sqrt(dx_pred**2 + dy_pred**2 + 1e-7)   # <-- add epsilon
    edge_loss = K.mean(K.abs(edge_true - edge_pred))
    return bce + d + 0.5*edge_loss


def iou_metric(y_true, y_pred, smooth=1):
    intersection = K.sum(K.abs(y_true * y_pred))
    union = K.sum(y_true) + K.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, BatchNormalization,
    Activation, concatenate, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def O_build_refined_unet(input_size=(256, 256, 3)):
    inputs = Input(input_size)
    encoder = ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
    encoder.trainable = False                         # <-- freeze encoder

    s1 = encoder.get_layer('conv1_relu').output
    s2 = encoder.get_layer('conv2_block3_out').output
    s3 = encoder.get_layer('conv3_block4_out').output
    s4 = encoder.get_layer('conv4_block6_out').output
    b  = encoder.get_layer('conv5_block3_out').output

    def dec_block(x, skip, filters):
        x = Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
        x = concatenate([x, skip])
        x = Conv2D(filters, 3, padding='same')(x)
        x = BatchNormalization()(x); x = Activation('relu')(x)
        x = Conv2D(filters, 3, padding='same')(x)
        x = BatchNormalization()(x); x = Activation('relu')(x)
        return x

    d1 = dec_block(b,  s4, 512)   # 16x16
    d2 = dec_block(d1, s3, 256)   # 32x32
    d3 = dec_block(d2, s2, 128)   # 64x64
    d4 = dec_block(d3, s1, 64)    # 128x128

    # one more upsample to get back to 256x256
    d5 = Conv2DTranspose(32, 2, strides=2, padding='same')(d4)
    d5 = Conv2D(32, 3, padding='same')(d5)
    d5 = BatchNormalization()(d5); d5 = Activation('relu')(d5)

    outputs = Conv2D(1, 1, activation='sigmoid')(d5)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(1e-3),                          # <-- higher LR for decoder only
        loss=bce_dice_edge_loss,
        metrics=[dice_coef, iou_metric],
    )
    return model



In [ ]:
# Training generators with augmentation
train_image_datagen = ImageDataGenerator(**image_datagen)
train_mask_datagen  = ImageDataGenerator(**mask_datagen)
#train_cusp_mask_datagen = ImageDataGenerator(**data_gen_args)

#train_image_datagen = ImageDataGenerator()
#train_mask_datagen  = ImageDataGenerator()
# Validation generators without augmentation
val_image_datagen   = ImageDataGenerator()
val_mask_datagen    = ImageDataGenerator()
#val_cusp_mask_datagen = ImageDataGenerator()

In [ ]:
# %%
train_gen = ResNetSegmentationDataGenerator(
    imgs_dir, masks_dir, batch_size,
    train_image_datagen, train_mask_datagen
)
val_gen   = ResNetSegmentationDataGenerator(
    val_imgs_dir, val_masks_dir, val_batch_size,
    val_image_datagen, val_mask_datagen
)

# Sanity checks
x_val, y_val = val_gen[0]
x_tr , y_tr  = train_gen[0]
print("Validation batch shapes:", x_val.shape, y_val.shape)
print("Training   batch shapes:", x_tr.shape , y_tr.shape)

### Train the occlusal decoder

In [ ]:
TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ─── TRAINING ──────────────────────────────────────────────────────────────────
model = O_build_refined_unet()
model.summary()


checkpoint = ModelCheckpoint(
    str(TRAINING_OUTPUT_DIR / "O_unet_refined.h5"), save_best_only=True,
    monitor="val_iou_metric", mode="max", verbose=1
)
earlystop = EarlyStopping(
    monitor="val_iou_metric", mode="max",
    patience=40, restore_best_weights=True, verbose=1
)
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor="val_iou_metric", mode="max",
    factor=0.5, patience=10, min_lr=1e-7, verbose=1
)

history = model.fit(
    train_gen, validation_data=val_gen,
    epochs=500,
    callbacks=[checkpoint, earlystop, reduce_lr]  # <-- add reduce_lr
)

### Cusp refined U-Net

### Defining directories for Cusp handling


In [ ]:
imgs_dir  = 'prep_O_M_image_folder_up'
masks_dir = 'prep_O_M_mask_folder_up'
cusp_masks_dir = 'prep_O_M_cusp_mask_folder_up'
val_imgs_dir  = 'prep_O_M_image_folder_up_val'
val_masks_dir = 'prep_O_M_mask_folder_up_val'
val_cusp_masks_dir = 'prep_O_M_CU_mask_folder_up_val'
samples_dir    = 'samples_stu_O'
# Output folder for predicted masks
pred_cusp_mask_dir = 'pred_O_M_cusp_molar_mask'
# Create output dir if needed
os.makedirs(pred_cusp_mask_dir, exist_ok=True)

### Cusp Data Augmentation and Instantiate generators for cusp mask Training and validation


In [ ]:
batch_size = 4
val_batch_size= 4
data_gen_args = dict(
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    shear_range=5,                 # degrees
    horizontal_flip=True,
    vertical_flip=True,
)

# Training generators with augmentation
train_image_datagen = ImageDataGenerator(**data_gen_args)
train_cusp_mask_datagen = ImageDataGenerator(**data_gen_args)

#train_image_datagen = ImageDataGenerator()
#train_mask_datagen  = ImageDataGenerator()
# Validation generators without augmentation
val_image_datagen   = ImageDataGenerator()
val_cusp_mask_datagen = ImageDataGenerator()

train_gen_cusp = UnitScaleSegmentationDataGenerator(
    imgs_dir, cusp_masks_dir, batch_size,
    train_image_datagen, train_cusp_mask_datagen
)
val_gen_cusp   = UnitScaleSegmentationDataGenerator(
    val_imgs_dir, val_cusp_masks_dir, val_batch_size,
    val_image_datagen, val_cusp_mask_datagen
)


# Sanity checks
x_val, y_val = val_gen_cusp[0]
x_tr , y_tr  = train_gen_cusp[0]
print("Validation cusp batch shapes:", x_val.shape, y_val.shape)
print("Training  cusp batch shapes:", x_tr.shape , y_tr.shape)

### Define Refined U-Net segmentation Model for CUSP mask prediction


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, MaxPooling2D,
    Dropout, concatenate, BatchNormalization, Activation
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import backend as K

# ─── CUSTOM LOSSES & METRICS ───────────────────────────────────────────────────
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.*intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

#def bce_dice_edge_loss(y_true, y_pred):
#    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
#    d   = dice_loss(y_true, y_pred)
#    # Sobel edge loss
#    gy_true, gx_true = tf.image.sobel_edges(y_true)[...,0], tf.image.sobel_edges(y_true)[...,1]
#    gy_pred, gx_pred = tf.image.sobel_edges(y_pred)[...,0], tf.image.sobel_edges(y_pred)[...,1]
#    edge_true = tf.sqrt(gx_true**2 + gy_true**2)
#    edge_pred = tf.sqrt(gx_pred**2 + gy_pred**2)
#    edge_loss = K.mean(K.abs(edge_true - edge_pred))
#    return bce + d + 0.5*edge_loss
def bce_dice_edge_loss(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    d   = dice_loss(y_true, y_pred)

    # sobel_edges expects 4D (B,H,W,C); returns (B,H,W,C,2) where last axis = [dy, dx]
    sob_true = tf.image.sobel_edges(y_true)
    sob_pred = tf.image.sobel_edges(y_pred)

    dy_true, dx_true = sob_true[..., 0], sob_true[..., 1]
    dy_pred, dx_pred = sob_pred[..., 0], sob_pred[..., 1]

    # +1e-6 inside sqrt keeps the gradient finite at zero-gradient regions
    edge_true = tf.sqrt(dx_true**2 + dy_true**2 + 1e-6)
    edge_pred = tf.sqrt(dx_pred**2 + dy_pred**2 + 1e-6)

    edge_loss = K.mean(K.abs(edge_true - edge_pred))
    return bce + d + 0.5 * edge_loss


def iou_metric(y_true, y_pred, smooth=1):
    intersection = K.sum(K.abs(y_true * y_pred))
    union = K.sum(y_true) + K.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

# ─── REFINED U-NET ARCHITECTURE ────────────────────────────────────────────────
def CU_build_refined_unet(input_size=(256,256,3)):
    inputs = Input(input_size)
    def enc_block(x, filters):
        x = Conv2D(filters,3,padding="same")(x)
        x = BatchNormalization()(x); x = Activation("relu")(x)
        x = Conv2D(filters,3,padding="same")(x)
        x = BatchNormalization()(x); x = Activation("relu")(x)
        p = MaxPooling2D()(x)
        return x, p

    c1,p1 = enc_block(inputs,64)
    c2,p2 = enc_block(p1,128)
    c3,p3 = enc_block(p2,256)
    c4,p4 = enc_block(p3,512)

    b = Conv2D(1024,3,padding="same")(p4)
    b = BatchNormalization()(b); b = Activation("relu")(b)
    b = Dropout(0.5)(b)
    b = Conv2D(1024,3,padding="same")(b)
    b = BatchNormalization()(b); b = Activation("relu")(b)

    def dec_block(x, skip, filters):
        x = Conv2DTranspose(filters,2,strides=2,padding="same")(x)
        x = concatenate([x,skip])
        x = Conv2D(filters,3,padding="same")(x)
        x = BatchNormalization()(x); x = Activation("relu")(x)
        x = Conv2D(filters,3,padding="same")(x)
        x = BatchNormalization()(x); x = Activation("relu")(x)
        return x

    d1 = dec_block(b, c4, 512)
    d2 = dec_block(d1, c3, 256)
    d3 = dec_block(d2, c2, 128)
    d4 = dec_block(d3, c1,  64)

    outputs = Conv2D(1,1,activation="sigmoid")(d4)
    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(1e-4),
        loss=bce_dice_edge_loss,
        metrics=[dice_coef, iou_metric]
    )
    return model

### Fitting Refined U-Net segmentation Model for CUSP prediction


In [ ]:
TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ─── TRAINING ──────────────────────────────────────────────────────────────────
model = CU_build_refined_unet()
model.summary()


checkpoint = ModelCheckpoint(
    str(TRAINING_OUTPUT_DIR / "CU_unet_refined.h5"), save_best_only=True,
    monitor="val_iou_metric", mode="max", verbose=1
)
earlystop = EarlyStopping(
    monitor="val_iou_metric", mode="max",
    patience=20, restore_best_weights=True, verbose=1
)

history = model.fit(
     train_gen_cusp,
     validation_data=val_gen_cusp,
     epochs=500,
     callbacks=[checkpoint, earlystop]
 )

## 5. Generate measurements and result figures

Continue with the packaged scripts for the final calculations. Their CLI
options specify the original inputs and output folders.

| Result | Script |
| --- | --- |
| Occlusal/proximal EFD scores and contours | [final_avg_efd_score.py](final_avg_efd_score.py) |
| Isthmus detection and widths | [isthmus_disc.py](isthmus_disc.py) |
| Cusp pairing, distances and isthmus ratios | [cus_ist_ratio.py](cus_ist_ratio.py) and [cusp_geometry.py](cusp_geometry.py) |
| Cavity floor detection and measurement | [landmark_cav_con_comb_smooth.py](landmark_cav_con_comb_smooth.py) |
| Depth and regularity PNG/HTML figures | [publication_cavity_depth.py](publication_cavity_depth.py) |
| CQS scores and tables | [final_CQS_overall.py](final_CQS_overall.py) |
| Filtered result presentation | [export_result_final.py](export_result_final.py) |

[Package instructions](README.md) describe external mesh, landmark and
reference-mask inputs, plus the unavailable refined-detector source dependency.